In [18]:
# Import libraries and load the 6 data files
import pandas as pd
import numpy as np

# low_memory=False is used to avoid the warning about mixed data types

# EIA files (outages, smart meters, circuits)
reliability = pd.read_csv('core_eia861__yearly_reliability.csv')
ami = pd.read_csv('core_eia861__yearly_advanced_metering_infrastructure.csv', low_memory=False)
dist = pd.read_csv('core_eia861__yearly_distribution_systems.csv')

# FERC files (line dollars, maintenance spending)
plant = pd.read_csv('core_ferc1__yearly_plant_in_service_sched204.csv', low_memory=False)
opex = pd.read_csv('core_ferc1__yearly_operating_expenses_sched320.csv', low_memory=False)

# PUDL crosswalk (Will be used to merge the FERC and EIA data using one ID)
crosswalk = pd.read_csv('utility_id_pudl.csv')

# Check each file (rows, columns)
for name, df in [('reliability', reliability), ('ami', ami), ('dist', dist),
                 ('plant', plant), ('opex', opex), ('crosswalk', crosswalk)]:
    print(name, df.shape)

reliability (26310, 22)
ami (158688, 16)
dist (17013, 8)
plant (298857, 14)
opex (508878, 8)
crosswalk (17724, 6)


In [19]:
# Keep only investor-owned utilities, 2021-2025, IEEE standard

# Creates a year column from the report date since the data in the report date column is in 2013-01-01T00:00:00.000Z format
reliability['year'] = pd.to_datetime(reliability['report_date']).dt.year

# Keeps only the rows we want from the columns listed to the rel variable
rel = reliability[
    (reliability['entity_type'] == 'Investor Owned') & # Keeps only investor-owned utilities
    (reliability['year'].between(2021, 2025)) & # Keeps only 2021-2025
    (reliability['standard'] == 'ieee_standard') # Keeps only if the standard is IEEE
]

# Keeps only the columns we want from the columns listed to the rel variable
rel = rel[['utility_id_eia', 'utility_name_eia', 'state', 'year', 'customers',
           'saidi_wo_major_event_days_minutes', 'saidi_w_major_event_days_minutes']]

# Only drops rows with no SAIDI value
rel = rel.dropna(subset=['saidi_wo_major_event_days_minutes'])

# Checks
print(f"Rows: {len(rel):,}")
print(f"Utilities: {rel['utility_id_eia'].nunique()}")
print(f"Duplicate utility-state-year rows: {rel.duplicated(['utility_id_eia', 'state', 'year']).sum()}")
print(rel.head())
print(rel['saidi_wo_major_event_days_minutes'].describe())

Rows: 705
Utilities: 114
Duplicate utility-state-year rows: 0
       utility_id_eia                   utility_name_eia state  year  \
16660             213  Alaska Electric Light & Power Co.    AK  2021   
16698             733               Appalachian Power Co    VA  2021   
16700             733               Appalachian Power Co    WV  2021   
16702             803          Arizona Public Service Co    AZ  2021   
16704             814               Entergy Arkansas LLC    AR  2021   

       customers  saidi_wo_major_event_days_minutes  \
16660    17184.0                              87.00   
16698   542478.0                             351.30   
16700   421594.0                             572.20   
16702  1314670.0                              87.36   
16704   738396.0                             263.30   

       saidi_w_major_event_days_minutes  
16660                            147.00  
16698                            566.60  
16700                           2087.60  
16702 

In [20]:
# Build smart meter share from the AMI file and merge it onto rel data frame

# Create a year column in AMI data frame
ami['year'] = pd.to_datetime(ami['report_date']).dt.year

# Keep investor-owned utilities, 2021-2025
ami_f = ami[(ami['entity_type'] == 'Investor Owned') & (ami['year'].between(2021, 2025))] 

# Add up meters across customer classes for each utility-state-year (AMI = new, AMR = old and read by receiver in a passing truck, non_amr_ami = old and read by person manually)
meter_cols = ['advanced_metering_infrastructure', 'automated_meter_reading', 'non_amr_ami']

# Collapse the customer-class rows: group by utility + state + year, add up each meter type,
# then reset_index() turns the group keys back into normal columns so we can merge on them
ami_sum = ami_f.groupby(['utility_id_eia', 'state', 'year'])[meter_cols].sum().reset_index()

# Add the 3 meter types across each row (axis=1) to get total meters
ami_sum['total_meters'] = ami_sum[meter_cols].sum(axis=1)
ami_sum['smart_meter_share'] = ami_sum['advanced_metering_infrastructure'] / ami_sum['total_meters']

# Attach smart_meter_share to the reliability table, matching on utility + state + year.
# how='left' keeps every rel row even if no AMI match (missing ones show as NaN)
df = rel.merge(ami_sum[['utility_id_eia', 'state', 'year', 'smart_meter_share']],
               on=['utility_id_eia', 'state', 'year'], how='left')

# check the result
print(f"AMI rows after filter: {len(ami_f):,}")
print(f"AMI rows after summing classes: {len(ami_sum):,}")
print(f"Rows in df after merge: {len(df):,}")
print(f"Rows missing smart meter share: {df['smart_meter_share'].isna().sum()}")
print(df['smart_meter_share'].describe())

AMI rows after filter: 3,880
AMI rows after summing classes: 960
Rows in df after merge: 705
Rows missing smart meter share: 0
count    705.000000
mean       0.662235
std        0.440479
min        0.000000
25%        0.013835
50%        0.989300
75%        0.998061
max        1.000000
Name: smart_meter_share, dtype: float64


In [21]:
# Add circuit counts and calculate customers per circuit

# create a year column
dist['year'] = pd.to_datetime(dist['report_date']).dt.year

# keep 2021-2025
dist_f = dist[dist['year'].between(2021, 2025)]

# merge circuit counts onto df
df = df.merge(dist_f[['utility_id_eia', 'state', 'year', 'distribution_circuits']],
              on=['utility_id_eia', 'state', 'year'], how='left')

# calculate customers per circuit
df['customers_per_circuit'] = df['customers'] / df['distribution_circuits']

# check the result
print(f"Rows in df after merge: {len(df):,}")
print(f"Rows missing circuits: {df['distribution_circuits'].isna().sum()}")
print(f"Rows with 0 circuits: {(df['distribution_circuits'] == 0).sum()}")
print(df['customers_per_circuit'].describe())

Rows in df after merge: 705
Rows missing circuits: 0
Rows with 0 circuits: 0
count     705.000000
mean      883.111815
std       434.192728
min         7.500000
25%       642.575178
50%       838.127072
75%      1081.247475
max      3692.925373
Name: customers_per_circuit, dtype: float64


In [22]:
# Build underground share from the FERC plant-in-service file

# keep 2021-2025 and only the 4 power line accounts
line_accounts = ['364', '365', '366', '367']
plant_f = plant[(plant['report_year'].between(2021, 2025)) &
                (plant['ferc_account'].astype(str).isin(line_accounts))]

# pivot so each company-year is 1 row, with 1 column per account
lines = plant_f.pivot_table(index=['utility_id_ferc1', 'report_year'],
                            columns='ferc_account',
                            values='ending_balance',
                            aggfunc='sum').reset_index()

# remove the column name that pandas creates
lines.columns.name = None

# treat accounts a company didn't report as $0
lines[line_accounts] = lines[line_accounts].fillna(0)

# add up overhead and underground dollars, then calculate the share
lines['overhead_dollars'] = lines['364'] + lines['365']
lines['underground_dollars'] = lines['366'] + lines['367']
lines['underground_share'] = lines['underground_dollars'] / (lines['overhead_dollars'] + lines['underground_dollars'])

# check the result
print(f"Plant rows after filter: {len(plant_f):,}")
print(f"Company-years after pivot: {len(lines):,}")
print(f"Company-years with $0 in all 4 accounts: {(lines['overhead_dollars'] + lines['underground_dollars'] == 0).sum()}")
print(lines.head())
print(lines['underground_share'].describe())

Plant rows after filter: 2,815
Company-years after pivot: 720
Company-years with $0 in all 4 accounts: 37
   utility_id_ferc1  report_year           364           365           366  \
0                92         2022  0.000000e+00  0.000000e+00  0.000000e+00   
1                92         2023  0.000000e+00  0.000000e+00  0.000000e+00   
2               152         2025  0.000000e+00  0.000000e+00  0.000000e+00   
3               153         2025  0.000000e+00  0.000000e+00  0.000000e+00   
4               155         2021  4.363508e+09  2.929840e+09  3.039193e+09   

            367  overhead_dollars  underground_dollars  underground_share  
0  0.000000e+00      0.000000e+00         0.000000e+00                NaN  
1  0.000000e+00      0.000000e+00         0.000000e+00                NaN  
2  0.000000e+00      0.000000e+00         0.000000e+00                NaN  
3  0.000000e+00      0.000000e+00         0.000000e+00                NaN  
4  7.106391e+09      7.293348e+09         1.0

In [23]:
# Total line maintenance spending per company-year

# keep 2021-2025 and only the 2 line maintenance expense types
maint_types = ['maintenance_of_overhead_lines', 'maintenance_of_underground_lines']
opex_f = opex[(opex['report_year'].between(2021, 2025)) &
              (opex['expense_type'].isin(maint_types))]

# check for duplicates
print(f"Opex rows after filter: {len(opex_f):,}")
print(f"Duplicate company-year-expense rows: {opex_f.duplicated(['utility_id_ferc1', 'report_year', 'expense_type']).sum()}")

# add overhead + underground maintenance into one total per company-year
maint = opex_f.groupby(['utility_id_ferc1', 'report_year'])['dollar_value'].sum().reset_index()

# give the column a clear name
maint = maint.rename(columns={'dollar_value': 'line_maintenance_dollars'})

# check the result
print(f"Company-years after summing: {len(maint):,}")
print(f"Company-years with negative maintenance: {(maint['line_maintenance_dollars'] < 0).sum()}")
print(maint.head())
print(maint['line_maintenance_dollars'].describe())

Opex rows after filter: 1,314
Duplicate company-year-expense rows: 0
Company-years after summing: 665
Company-years with negative maintenance: 0
   utility_id_ferc1  report_year  line_maintenance_dollars
0               155         2021               548612497.0
1               155         2022               643460846.0
2               155         2023               544231226.0
3               155         2024               655045401.0
4               155         2025               596264558.0
count    6.650000e+02
mean     7.885391e+07
std      1.916580e+08
min      0.000000e+00
25%      9.009408e+06
50%      3.740645e+07
75%      7.946047e+07
max      2.353368e+09
Name: line_maintenance_dollars, dtype: float64


In [24]:
# Combine the two FERC tables, attach EIA IDs with the crosswalk, merge onto df

# combine underground share and maintenance into one FERC table
ferc = lines[['utility_id_ferc1', 'report_year', 'underground_share']].merge(
    maint, on=['utility_id_ferc1', 'report_year'], how='left')

# clean the crosswalk down to just the two ID columns
xwalk = crosswalk[['utility_id_ferc1', 'utility_id_eia']].dropna().drop_duplicates()

# attach EIA IDs to the FERC table
ferc = ferc.merge(xwalk, on='utility_id_ferc1', how='inner')
ferc = ferc.rename(columns={'report_year': 'year'})
print(f"FERC company-years with an EIA ID: {len(ferc):,}")
print(f"Duplicate EIA ID-year rows: {ferc.duplicated(['utility_id_eia', 'year']).sum()}")

# merge FERC onto df by EIA ID + year (no state: FERC is company-wide)
rows_before = len(df)
df = df.merge(ferc[['utility_id_eia', 'year', 'underground_share', 'line_maintenance_dollars']],
              on=['utility_id_eia', 'year'], how='left')
print(f"Rows in df before merge: {rows_before:,} | after: {len(df):,}")
print(f"Rows missing underground share: {df['underground_share'].isna().sum()}")
print(f"Rows missing maintenance: {df['line_maintenance_dollars'].isna().sum()}")

# total customers per company-year (all states added together)
company_cust = reliability[(reliability['entity_type'] == 'Investor Owned') &
                           (reliability['year'].between(2021, 2025))]
company_cust = company_cust.groupby(['utility_id_eia', 'state', 'year'])['customers'].max().reset_index()
company_cust = company_cust.groupby(['utility_id_eia', 'year'])['customers'].sum().reset_index()
company_cust = company_cust.rename(columns={'customers': 'company_customers'})

# merge company customers onto df and calculate maintenance per customer
df = df.merge(company_cust, on=['utility_id_eia', 'year'], how='left')
df['maintenance_per_customer'] = df['line_maintenance_dollars'] / df['company_customers']
print(f"Rows in df: {len(df):,}")
print(df['maintenance_per_customer'].describe())

FERC company-years with an EIA ID: 629
Duplicate EIA ID-year rows: 0
Rows in df before merge: 705 | after: 705
Rows missing underground share: 123
Rows missing maintenance: 123
Rows in df: 705
count    582.000000
mean      82.198384
std       58.123979
min        1.716634
25%       45.745655
50%       67.645330
75%      104.426772
max      430.980158
Name: maintenance_per_customer, dtype: float64


In [25]:
# Drop incomplete rows, remove tiny service areas, save the clean dataset

# The columns the regression needs
model_cols = ['saidi_wo_major_event_days_minutes', 'underground_share', 'maintenance_per_customer',
              'smart_meter_share', 'customers_per_circuit']

# turn any infinity (from dividing by 0) into NaN
df = df.replace([np.inf, -np.inf], np.nan)
print(f"Missing values per column:\n{df[model_cols].isna().sum()}\n")

# drop rows missing any model variable
clean = df.dropna(subset=model_cols).copy()
print(f"Rows after dropping missing: {len(clean):,} (dropped {len(df) - len(clean):,})")

# look at, then remove, service areas with under 1,000 customers to make sure those dont skew data
tiny = clean[clean['customers'] < 1000]
print(f"\nRows with under 1,000 customers:\n{tiny[['utility_name_eia', 'state', 'year', 'customers', 'saidi_wo_major_event_days_minutes']]}\n")
clean = clean[clean['customers'] >= 1000]

# final summary
print(f"Final rows: {len(clean):,}")
print(f"Final utilities: {clean['utility_id_eia'].nunique()}")
print(f"Years: {sorted(clean['year'].unique().tolist())}\n")
print(clean[model_cols].describe().round(2).T)

# save for the analysis notebook
clean.to_csv('clean_utility_data.csv', index=False)
print("\nSaved clean_utility_data.csv")

Missing values per column:
saidi_wo_major_event_days_minutes      0
underground_share                    123
maintenance_per_customer             123
smart_meter_share                      0
customers_per_circuit                  0
dtype: int64

Rows after dropping missing: 582 (dropped 123)

Rows with under 1,000 customers:
            utility_name_eia state  year  customers  \
117  Black Hills Power, Inc.    MT  2021       40.0   
256  Black Hills Power, Inc.    MT  2022       41.0   
395  Black Hills Power, Inc.    MT  2023       30.0   
538  Black Hills Power, Inc.    MT  2024       43.0   
682  Black Hills Power, Inc.    MT  2025       47.0   

     saidi_wo_major_event_days_minutes  
117                             62.612  
256                            216.774  
395                            142.933  
538                             59.400  
682                            672.633  

Final rows: 577
Final utilities: 89
Years: [2021, 2022, 2023, 2024, 2025]



                                   count    mean     std     min     25%  \
saidi_wo_major_event_days_minutes  577.0  139.74  109.91    2.17   78.80   
underground_share                  577.0    0.38    0.16    0.03    0.27   
maintenance_per_customer           577.0   82.05   58.33    1.72   45.61   
smart_meter_share                  577.0    0.70    0.43    0.00    0.16   
customers_per_circuit              577.0  893.10  439.94  118.82  649.50   

                                      50%      75%      max  
saidi_wo_major_event_days_minutes  108.39   160.12   928.50  
underground_share                    0.39     0.46     0.90  
maintenance_per_customer            67.58   104.01   430.98  
smart_meter_share                    0.99     1.00     1.00  
customers_per_circuit              834.01  1077.01  3692.93  

Saved clean_utility_data.csv
